# 00 - Pipeline Orchestrator

**Purpose:** Server-side orchestration of all translation pipeline notebooks.

This notebook:
1. Reads pipeline configuration from `Files/config/pipeline_run.json`
2. Runs notebooks NB01-NB09 in sequence using `mssparkutils.notebook.run()`
3. Writes progress to `Files/config/pipeline_progress.json` after each step
4. Allows app to poll progress without keeping browser open

## Benefits
- Runs entirely server-side (no browser dependency)
- Progress visible even after browser closes
- Built-in error handling and status reporting

In [ ]:
import json
import os
import traceback
from datetime import datetime
from notebookutils import mssparkutils

In [ ]:
# Configuration
CONFIG_DIR = "/lakehouse/default/Files/config"
PROGRESS_FILE = f"{CONFIG_DIR}/pipeline_progress.json"
PIPELINE_CONFIG_FILE = f"{CONFIG_DIR}/pipeline_run.json"

# Timeout per notebook (1 hour default, can be overridden per notebook)
DEFAULT_TIMEOUT = 3600

# Pipeline steps - notebooks to run in order (NB00 is this orchestrator, tracked separately)
PIPELINE_STEPS = [
    {"id": "NB00", "name": "Initialize", "notebook": "00_pipeline_orchestrator", "timeout": 60},
    {"id": "NB01", "name": "Parse RDF", "notebook": "01_rdf_parser_jena", "timeout": 1800},
    {"id": "NB02", "name": "Detect Schema", "notebook": "02_schema_detector", "timeout": 600},
    {"id": "NB03", "name": "Map Classes", "notebook": "03_class_to_nodetype", "timeout": 600},
    {"id": "NB04", "name": "Map Properties", "notebook": "04_property_mapping", "timeout": 600},
    {"id": "NB05", "name": "Translate Instances", "notebook": "05_instance_translator", "timeout": 1800},
    {"id": "NB06", "name": "Write Delta Tables", "notebook": "06_delta_writer", "timeout": 1800},
    {"id": "NB07", "name": "Generate Ontology", "notebook": "07_ontology_definition_generator", "timeout": 600},
    {"id": "NB08", "name": "Push to API", "notebook": "08_ontology_api_client", "timeout": 1800},
    {"id": "NB09", "name": "Bind to Graph", "notebook": "09_data_binding", "timeout": 3600},
]

print(f"Pipeline has {len(PIPELINE_STEPS)} steps")

In [ ]:
def write_progress(current_step: str, completed: list, status: str, error: str = None, step_times: dict = None):
    """
    Write pipeline progress to JSON file for app polling.
    
    Args:
        current_step: ID of currently running step (e.g., 'NB03') or None if complete
        completed: List of completed step IDs
        status: 'running', 'completed', 'failed'
        error: Error message if failed
        step_times: Dict of {step_id: {start: iso, end: iso, duration_sec: int}}
    """
    os.makedirs(CONFIG_DIR, exist_ok=True)
    
    progress = {
        "current": current_step,
        "completed": completed,
        "status": status,
        "error": error,
        "step_times": step_times or {},
        "total_steps": len(PIPELINE_STEPS),
        "updated_at": datetime.now().isoformat(),
    }
    
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(progress, f, indent=2)
    
    print(f"Progress: {status} - {len(completed)}/{len(PIPELINE_STEPS)} steps complete")


def load_pipeline_config() -> dict:
    """
    Load pipeline configuration written by app.
    """
    if os.path.exists(PIPELINE_CONFIG_FILE):
        with open(PIPELINE_CONFIG_FILE, 'r') as f:
            return json.load(f)
    return {}


# Load config
pipeline_config = load_pipeline_config()
print(f"Pipeline config: {pipeline_config.get('project_name', 'unknown project')}")
print(f"Created by: {pipeline_config.get('created_by', 'unknown')}")

In [ ]:
def run_pipeline():
    """
    Execute all pipeline notebooks in sequence.
    Writes progress after each step for app polling.
    """
    completed = []
    step_times = {}
    
    print("="*60)
    print("Starting RDF Translation Pipeline")
    print("="*60)
    
    pipeline_start = datetime.now()
    
    # Mark NB00 (Initialize) as running immediately
    step_times["NB00"] = {"start": pipeline_start.isoformat()}
    write_progress(
        current_step="NB00",
        completed=[],
        status="running",
        step_times=step_times
    )
    
    # Mark NB00 as complete and move to first real step (NB01)
    nb00_end = datetime.now()
    step_times["NB00"]["end"] = nb00_end.isoformat()
    step_times["NB00"]["duration_sec"] = int((nb00_end - pipeline_start).total_seconds())
    completed.append("NB00")
    
    # Start processing NB01-NB09 (skip NB00 in the loop)
    for i, step in enumerate(PIPELINE_STEPS[1:]):  # Skip NB00
        step_id = step["id"]
        notebook_name = step["notebook"]
        timeout = step.get("timeout", DEFAULT_TIMEOUT)
        
        print(f"\n{'='*60}")
        print(f"Step {i+1}/{len(PIPELINE_STEPS)}: {step_id} - {step['name']}")
        print(f"Notebook: {notebook_name}")
        print(f"Timeout: {timeout}s")
        print("="*60)
        
        # Update progress - starting this step
        write_progress(
            current_step=step_id,
            completed=completed,
            status="running",
            step_times=step_times
        )
        
        step_start = datetime.now()
        
        try:
            # Run the notebook
            result = mssparkutils.notebook.run(notebook_name, timeout)
            
            step_end = datetime.now()
            duration = (step_end - step_start).total_seconds()
            
            # Record timing
            step_times[step_id] = {
                "start": step_start.isoformat(),
                "end": step_end.isoformat(),
                "duration_sec": int(duration)
            }
            
            completed.append(step_id)
            print(f"\n✓ {step_id} completed in {int(duration)}s")
            
        except Exception as e:
            step_end = datetime.now()
            duration = (step_end - step_start).total_seconds()
            
            step_times[step_id] = {
                "start": step_start.isoformat(),
                "end": step_end.isoformat(),
                "duration_sec": int(duration),
                "error": str(e)
            }
            
            error_msg = f"{step_id} failed: {str(e)}"
            print(f"\n✗ {error_msg}")
            traceback.print_exc()
            
            # Write failure progress
            write_progress(
                current_step=step_id,
                completed=completed,
                status="failed",
                error=error_msg,
                step_times=step_times
            )
            
            raise Exception(error_msg)
    
    # All steps completed
    pipeline_end = datetime.now()
    total_duration = (pipeline_end - pipeline_start).total_seconds()
    
    write_progress(
        current_step=None,
        completed=completed,
        status="completed",
        step_times=step_times
    )

    print(f"\n{'='*60}")    

    print("Pipeline Complete!")
    print("="*60)

    print(f"Total duration: {int(total_duration)}s ({int(total_duration/60)}m)")
    print(f"Steps completed: {len(completed)}/{len(PIPELINE_STEPS)}")
    
    return completed

In [ ]:
# Run the pipeline
try:
    completed_steps = run_pipeline()
    print(f"\nSuccessfully completed: {completed_steps}")
except Exception as e:
    print(f"\nPipeline failed: {e}")
    raise

In [ ]:
# Summary - read final progress
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, 'r') as f:
        final_progress = json.load(f)
    
    print("\nFinal Progress Report")
    print("="*60)
    print(f"Status: {final_progress['status']}")
    print(f"Steps completed: {len(final_progress['completed'])}/{final_progress['total_steps']}")
    
    print("\nStep Timings:")
    for step_id, times in final_progress.get('step_times', {}).items():
        duration = times.get('duration_sec', 0)
        status = '✓' if step_id in final_progress['completed'] else '✗'
        print(f"  {status} {step_id}: {duration}s")
else:
    print("No progress file found")